# Model Implementation 
### SALSABILA AURELIA OKTANIOPUTRI TP078641

In [2]:
!pip install prophet

     ---------------------------------------- 0.0/53.2 kB ? eta -:--:--
     ---------------------------------------- 53.2/53.2 kB 1.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/12.1 MB 4.1 MB/s eta 0:00:03
    --------------------------------------- 0.2/12.1 MB 2.1 MB/s eta 0:00:06
    --------------------------------------- 0.3/12.1 MB 2.2 MB/s eta 0:00:06
   - -------------------------------------- 0.4/12.1 MB 2.3 MB/s eta 0:00:06
   - -------------------------------------- 0.6/12.1 MB 2.5 MB/s eta 0:00:05
   -- ------------------------------------- 0.7/12.1 MB 2.6 MB/s eta 0:00:05
   -- ------------------------------------- 0.9/12.1 MB 2.7 MB/s eta 0:00:05
   --- ------------------------------------ 1.0/12.1 MB 2.7 MB/s eta 0:00:05
   --- ------------------------------------ 1.1/12.1 MB 2.8 MB/s eta 0:00:04
   ---- ----------------------------------- 1.2/12.1 MB 2.7 MB/s eta 0:00:04
   ---- 


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
!pip install plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
    --------------------------------------- 0.2/9.9 MB 4.1 MB/s eta 0:00:03
   - -------------------------------------- 0.4/9.9 MB 4.1 MB/s eta 0:00:03
   - -------------------------------------- 0.5/9.9 MB 3.3 MB/s eta 0:00:03
   - -------------------------------------- 0.5/9.9 MB 3.3 MB/s eta 0:00:03
   --- ------------------------------------ 0.9/9.9 MB 4.4 MB/s eta 0:00:03
   ---- ----------------------------------- 1.2/9.9 MB 4.4 MB/s eta 0:00:03
   ----- ---------------------------------- 1.5/9.9 MB 4.7 MB/s eta 0:00:02
   ------ --------------------------------- 1.7/9.9 MB 4.6 MB/s eta 0:00:02
   ------- -------------------------------- 1.9/9.9 MB 4.6 MB/s eta 0:00:02
   -------- ------------------------------- 2.0/9.9 MB 4.5 MB/s eta 0:00:02
   --------- ------------------------------ 2.3/9.9 MB 4.6 MB/s eta 0:00:02
   ---------- ----------------------------- 2.5/9.9 MB 4.6 MB/s eta 0:00:02
   ----------- ----


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\asus\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet

plt.rcParams['figure.figsize'] = (14, 5)

# Reload cleaned data, fixing datetime dtype lost in CSV
df_weekly_clean = pd.read_csv('data/df_weekly_clean.csv', parse_dates=['ds'])
train = pd.read_csv('data/train.csv', parse_dates=['ds'])
val   = pd.read_csv('data/val.csv', parse_dates=['ds'])
test  = pd.read_csv('data/test.csv', parse_dates=['ds'])

print(f"Full series : {df_weekly_clean.shape}")
print(f"Train       : {train.shape}, {train['ds'].min().date()} to {train['ds'].max().date()}")
print(f"Validation  : {val.shape}, {val['ds'].min().date()} to {val['ds'].max().date()}")
print(f"Test        : {test.shape}, {test['ds'].min().date()} to {test['ds'].max().date()}")

Full series : (209, 2)
Train       : (146, 2), 2020-01-05 to 2022-10-16
Validation  : (21, 2), 2022-10-23 to 2023-03-12
Test        : (42, 2), 2023-03-19 to 2023-12-31


In [ ]:
# 2) SARIMA CHECK 1:
# tests for stationarity on the time series because sarima mathematically needs stationarity
# a non stationary series will need SARIMA to internally differences the data to remove the trend before fitting
# by testing we explicitly know how much differencing (d) to apply

def adf_test(series, label=''):
    result = adfuller(series.dropna())
    print(f"--- ADF Test: {label} ---")
    print(f"ADF Statistic : {result[0]:.4f}")
    print(f"p-value       : {result[1]:.4f}")
    if result[1] <= 0.05:
        print("=> Stationary (reject H0)")
    else:
        print("=> Non-stationary (fail to reject H0)")
    print()

# Test on raw training series
adf_test(train['y'], 'Original training series')

# Apply first-order differencing
train_diff1 = train['y'].diff().dropna()
adf_test(train_diff1, 'After 1st-order differencing (d=1)')

# Apply seasonal differencing (lag 52) on top, if still non-stationary
train_diff_seasonal = train['y'].diff().diff(52).dropna()
adf_test(train_diff_seasonal, 'After d=1 + seasonal D=1 (lag 52)')

- tests for stationarity on the time series because sarima mathematically needs stationarity
- a non stationary series will need SARIMA to internally differences the data to remove the trend before fitting
- by testing we explicitly know how much differencing (d) to apply

check
- look at the printed p values in order
- raw series should show pval > 0.05 (means non stationary)
- after d=1 p value should drop below 0.05 
- if it doesnt we may need d=2 but be carefull about over differencing


In [ ]:
# 3) SARIMA CHECK 2: ACF/PACF Analysis
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(train_diff1, lags=60, ax=axes[0])
axes[0].set_title('ACF — after differencing (helps identify q, Q)')
plot_pacf(train_diff1, lags=60, ax=axes[1])
axes[1].set_title('PACF — after differencing (helps identify p, P)')
plt.tight_layout()
plt.show()

In [ ]:
# 4) PROPHET Configuration
model_prophet = Prophet(
    seasonality_mode='multiplicative',   # because seasonal amplitude grows with the trend (seen in EDA3/EDA6)
    yearly_seasonality=True,
    weekly_seasonality=False,            # not meaningful at weekly aggregation level
    daily_seasonality=False,
    interval_width=0.95                  # MATCH SARIMA's 95% confidence interval — critical for fair comparison later
)

In [ ]:
# 5) Define Forecast Horizon
FORECAST_HORIZON = len(test)
print(f"Forecast horizon: {FORECAST_HORIZON} weeks (single multi-step forecast)")

In [ ]:
# 6) Model Fitting (Training data)
# SARIMA — use ADF/ACF/PACF findings as STARTING values; final values chosen in Step 6
sarima_order = (1, 1, 1)              # (p, d, q) — placeholder, refined in tuning
sarima_seasonal_order = (1, 1, 1, 52) # (P, D, Q, S) — placeholder, refined in tuning

model_sarima = SARIMAX(
    train['y'],
    order=sarima_order,
    seasonal_order=sarima_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = model_sarima.fit(disp=False)
print(sarima_fit.summary())

# Prophet requires columns named exactly 'ds' and 'y' — already matches your naming
model_prophet.fit(train[['ds', 'y']])
print("Prophet fitted on training data.")

In [ ]:
# 7) Hyperparameter Tuning 1: SARIMA
from sklearn.metrics import mean_squared_error
from itertools import product

# ── SARIMA grid search ─────────────────────────────────────────────
p_vals = [0, 1, 2]
d_vals = [1]              # locked in from ADF test result
q_vals = [0, 1, 2]
P_vals = [0, 1]
D_vals = [1]              # locked in from seasonal differencing result
Q_vals = [0, 1]
S = 52

best_rmse = np.inf
best_params = None
results_log = []

for p, d, q, P, D, Q in product(p_vals, d_vals, q_vals, P_vals, D_vals, Q_vals):
    try:
        m = SARIMAX(train['y'], order=(p,d,q), seasonal_order=(P,D,Q,S),
                    enforce_stationarity=False, enforce_invertibility=False)
        fit = m.fit(disp=False)
        forecast = fit.get_forecast(steps=len(val))
        pred = forecast.predicted_mean
        rmse = np.sqrt(mean_squared_error(val['y'], pred))
        results_log.append((p,d,q,P,D,Q, fit.aic, rmse))
        if rmse < best_rmse:
            best_rmse = rmse
            best_params = (p,d,q,P,D,Q)
    except Exception as e:
        continue

print(f"Best SARIMA params (p,d,q,P,D,Q): {best_params}")
print(f"Best validation RMSE: {best_rmse:.2f}")

results_df = pd.DataFrame(results_log, columns=['p','d','q','P','D','Q','AIC','val_RMSE'])
results_df.sort_values('val_RMSE').head(10)

In [ ]:
# 8) Hyperparameter Tuning 3: Prophet 
# ── Prophet tuning ──────────────────────────────────────────────────
changepoint_scales = [0.001, 0.01, 0.05, 0.1, 0.5]
seasonality_scales = [0.01, 1.0, 10.0]

best_prophet_rmse = np.inf
best_prophet_params = None
prophet_results_log = []

for cps, sps in product(changepoint_scales, seasonality_scales):
    m = Prophet(
        seasonality_mode='multiplicative',
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95,
        changepoint_prior_scale=cps,
        seasonality_prior_scale=sps
    )
    m.fit(train[['ds', 'y']])
    future_val = val[['ds']]
    fcst = m.predict(future_val)
    rmse = np.sqrt(mean_squared_error(val['y'], fcst['yhat']))
    prophet_results_log.append((cps, sps, rmse))
    if rmse < best_prophet_rmse:
        best_prophet_rmse = rmse
        best_prophet_params = (cps, sps)

print(f"Best Prophet params (changepoint, seasonality): {best_prophet_params}")
print(f"Best validation RMSE: {best_prophet_rmse:.2f}")

In [ ]:
# 9) Refit Model on Training Data with Best Hyperparameters
p,d,q,P,D,Q = best_params
final_sarima = SARIMAX(train['y'], order=(p,d,q), seasonal_order=(P,D,Q,52),
                        enforce_stationarity=False, enforce_invertibility=False)
final_sarima_fit = final_sarima.fit(disp=False)

cps, sps = best_prophet_params
final_prophet = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.95,
    changepoint_prior_scale=cps,
    seasonality_prior_scale=sps
)
final_prophet.fit(train[['ds', 'y']])

print(f"Final SARIMA order: ({p},{d},{q})({P},{D},{Q},52)")
print(f"Final Prophet params: changepoint_prior_scale={cps}, seasonality_prior_scale={sps}")

In [ ]:
# 10) Forecast Generation
# SARIMA forecast
sarima_forecast_obj = final_sarima_fit.get_forecast(steps=FORECAST_HORIZON)
sarima_pred = sarima_forecast_obj.predicted_mean
sarima_ci = sarima_forecast_obj.conf_int(alpha=0.05)  # 95% CI, matches Prophet's interval_width=0.95

sarima_results = pd.DataFrame({
    'ds': test['ds'].values,
    'actual': test['y'].values,
    'predicted': sarima_pred.values,
    'lower_ci': sarima_ci.iloc[:, 0].values,
    'upper_ci': sarima_ci.iloc[:, 1].values
})

# Prophet forecast
future_test = test[['ds']]
prophet_fcst = final_prophet.predict(future_test)

prophet_results = pd.DataFrame({
    'ds': test['ds'].values,
    'actual': test['y'].values,
    'yhat': prophet_fcst['yhat'].values,
    'yhat_lower': prophet_fcst['yhat_lower'].values,
    'yhat_upper': prophet_fcst['yhat_upper'].values,
    'trend': prophet_fcst['trend'].values,
    'seasonal': prophet_fcst['multiplicative_terms'].values
})

print("SARIMA results:")
print(sarima_results.head())
print("\nProphet results:")
print(prophet_results.head())

In [ ]:
# 11) Check Alignment
assert (sarima_results['ds'].values == prophet_results['ds'].values).all(), "Date mismatch between models!"
print("Date alignment confirmed — both models forecast the exact same weeks.")

In [ ]:
# 12) Export Results to Dashboard
sarima_results.to_csv('model_sarima_results.csv', index=False)
prophet_results.to_csv('model_prophet_results.csv', index=False)

print("Saved: model_sarima_results.csv")
print("Saved: model_prophet_results.csv")
print(f"\nForecast type used: SINGLE multi-step forecast, {FORECAST_HORIZON} weeks, no rolling/intermediate actuals used.")